In [ ]:
import pandas as pd
import optuna
import numpy as np
from sklearn.metrics import classification_report, f1_score

In [ ]:
pred_path = "../../logs/conll/test_preds/"


In [ ]:
dfs = []
for i in range (0, 5):
    df = pd.read_pickle(pred_path + f"val_data_{i}.pkl")
    dfs.append(df)

In [ ]:
def extract_model_data(dfs):
    # Get probabilities from each dataframe and make list of numpy arrays
    model_probs = []
    for i, df in enumerate(dfs):
        probs = np.array(df['Probabilities'].tolist())
        model_probs.append(probs)

    # Extract labels and make numpy array
    true_labels = None
    if 'Label' in dfs[0].columns:
        true_labels = np.array(dfs[0]['Label'].tolist())

    return model_probs, true_labels

model_probs, true_labels = extract_model_data(dfs)

In [ ]:
def combine_with_weights(model_probs, weights=None):
    # Set default weights if none provided
    if weights is None:
        weights = np.ones(len(model_probs)) / len(model_probs)

    # Validate weights
    assert len(weights) == len(model_probs), "Number of weights must match number of models"

    # Normalize weights to sum to 1
    weights = np.array(weights) / np.sum(weights)

    # Combine probabilities using weights
    ensemble_probs = np.zeros_like(model_probs[0])
    for i, probs in enumerate(model_probs):
        ensemble_probs += weights[i] * probs

    # Get predicted classes
    ensemble_preds = np.argmax(ensemble_probs, axis=1)

    return ensemble_probs, ensemble_preds

ensemble_probs, ensemble_preds = combine_with_weights(model_probs, [1, 1, 1, 1, 1])

In [ ]:
f1 = f1_score(true_labels, ensemble_preds, average='macro')
print(f'Macro f1 for ensemble (unweighted): {f1}')
avg_of_f1 = 0
for i, df in enumerate(dfs):
    y_true = df["Label"]
    y_pred = df["Prediction"]
    f1_macro = f1_score(y_true, y_pred, average='macro')
    avg_of_f1 += f1_macro
    print(f'Macro f1 for ensemble model {i}: {f1_macro}')

avg_of_f1 /= len(dfs)
print(f'Improvement over average: {f1 - avg_of_f1}')


In [ ]:
def optimize_ensemble_weights(model_probs, true_labels, n_trials=100):
    def objective(trial):
        # Generate weights using Optuna
        weights = [trial.suggest_float(f'weight_model_{i}', 0.0, 1.0) for i in range(len(model_probs))]

        # Use existing function to create weighted ensemble
        _, ensemble_preds = combine_with_weights(model_probs, weights)

        # Calculate F1 score
        f1 = f1_score(true_labels, ensemble_preds, average='macro')

        return -f1  # Negative because Optuna minimizes

    # Create and run the study
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    # Get best weights
    best_params = study.best_params
    best_weights = np.array([best_params[f'weight_model_{i}'] for i in range(len(model_probs))])
    best_weights = best_weights / np.sum(best_weights)  # Normalize

    # Get best f1
    best_f1 = -study.best_value

    return best_weights, best_f1


In [ ]:
n_trials = 500
best_weights, best_f1 = optimize_ensemble_weights(model_probs, true_labels, n_trials)

In [ ]:
print(f'Best weights: {best_weights}')
print(f'Best Macro F1: {best_f1}')

In [ ]:
ensemble_probs, ensemble_preds = combine_with_weights(model_probs, best_weights)


In [ ]:
f1 = f1_score(true_labels, ensemble_preds, average='macro')
print(f'Macro f1 for ensemble (weighted): {f1}')
avg_of_f1 = 0
for i, df in enumerate(dfs):
    y_true = df["Label"]
    y_pred = df["Prediction"]
    f1_macro = f1_score(y_true, y_pred, average='macro')
    avg_of_f1 += f1_macro
    print(f'Macro f1 for ensemble model {i}: {f1_macro}')

avg_of_f1 /= len(dfs)
print(f'Improvement over average: {f1 - avg_of_f1}')